# Build a Caskada research agent

This notebook builds the same three-node agent as the project source. Run each cell in order, then change the final question and observe the `search` loop and `answer` exit.

In [ ]:
%pip install -q "caskada>=3.0.0" "openai>=3.1.0" "duckduckgo-search>=8.1.1" "pyyaml>=6.0.3"

In [ ]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: " )

In [ ]:
import yaml
from caskada import Context, Flow, node
from duckduckgo_search import DDGS
from openai import OpenAI


def call_llm(prompt):
    response = OpenAI().chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content or ""


def search_web(query):
    results = DDGS().text(query, max_results=5)
    return "\n\n".join(
        f"Title: {item['title']}\nURL: {item['href']}\nSnippet: {item['body']}"
        for item in results
    )

In [ ]:
@node
def decide(context: Context):
    response = call_llm(f"""
You are a research assistant that can search the web.
Question: {context.state['question']}
Previous research: {context.state.get('research', 'No previous search')}

Return exactly one YAML code block:
```yaml
action: search
search_query: what to search for
```
or:
```yaml
action: answer
answer: final answer
```
""")
    if "```yaml" not in response:
        raise ValueError("decision must contain a YAML block")
    decision = yaml.safe_load(response.split("```yaml", 1)[1].split("```", 1)[0])
    if not isinstance(decision, dict) or decision.get("action") not in {"search", "answer"}:
        raise ValueError("decision action must be search or answer")
    if decision["action"] == "search":
        query = decision.get("search_query")
        if not isinstance(query, str) or not query.strip():
            raise ValueError("search decision needs a non-empty search_query")
        context.state["search_query"] = query
    else:
        answer = decision.get("answer")
        if not isinstance(answer, str) or not answer.strip():
            raise ValueError("answer decision needs a non-empty answer")
        context.state["research"] = answer
    context.emit(decision["action"])


@node
def search(context: Context):
    query = context.state["search_query"]
    results = search_web(query)
    context.state["research"] = (
        context.state.get("research", "") + f"\n\nSEARCH: {query}\nRESULTS: {results}"
    )
    context.emit("decide")


@node
def answer(context: Context):
    context.state["answer"] = call_llm(
        f"Question: {context.state['question']}\nResearch: {context.state.get('research', '')}"
    )

In [ ]:
decide.link(search, "search")
decide.link(answer, "answer")
search.link(decide, "decide")
agent = Flow(decide)

In [ ]:
state = await agent.run({"question": "Who won the Nobel Prize in Physics 2024?"})
print(state["answer"])